# TP2 -- Régressions Linéaires

**Statistique Mathématique 3 -- L3 MIASHS**

---

## Introduction

On étudie la relation entre la **tension artérielle** et l'**âge** des individus. Données simulées à partir de *BOUYER et al. (1995)*.

**Objectif :** Déterminer si l'âge influence la tension artérielle et prédire la tension en fonction de l'âge.

---
## 1. Régression Linéaire Simple

### 1.1 Statistiques descriptives

- **Variable explicative** (indépendante) : âge ($x$)
- **Variable expliquée** (dépendante) : tension ($y$)

In [ ]:
# Importation des données
data = read.csv("tension.csv", header = TRUE, sep = ";", dec = ",")
head(data)
x = data$age
y = data$tension

In [ ]:
# Nuage de points
plot(x, y, main = "Tension artérielle en fonction de l'âge",
     xlab = "Âge", ylab = "Tension (mmHg)", pch = 19, col = "steelblue")

In [ ]:
# Coefficient de corrélation
r = cor(x, y)
cat("Coefficient de corrélation r =", round(r, 4))

In [ ]:
# Coefficients par moindres carrés
beta1 = cov(x, y) / var(x)
beta0 = mean(y) - beta1 * mean(x)
cat("beta1 =", round(beta1, 4), "\nbeta0 =", round(beta0, 4))

In [ ]:
# Vérification avec lm
lm(y ~ x)

In [ ]:
# Tracé de la droite de régression
plot(x, y, main = "Régression linéaire : Tension ~ Âge",
     xlab = "Âge", ylab = "Tension", pch = 19, col = "steelblue")
abline(beta0, beta1, col = "red", lwd = 2)

### 1.2 Résidus

Les résidus mesurent l'écart entre les valeurs observées et prédites.

In [ ]:
# Calcul avec lm
model = lm(y ~ x)
beta0 = model$coefficients[1]
beta1 = model$coefficients[2]
yajustes = model$fitted.values
erreurs = model$residuals

In [ ]:
# Résidus vs valeurs ajustées
plot(yajustes, erreurs, main = "Résidus vs Valeurs ajustées",
     xlab = "Valeurs ajustées", ylab = "Résidus", pch = 19, col = "darkgreen")
abline(h = 0, col = "red", lty = 2)

**Pas de structure particulière** → c'est ce qu'on attend (erreurs $\sim \mathcal{N}(0, \sigma)$, indépendantes de $x$).

In [ ]:
# Densité des résidus vs gaussienne
plot(density(erreurs), main = "Densité des résidus", col = "blue", lwd = 2)
discr = -15:15
norm = dnorm(discr, 0, sqrt(var(erreurs)))
lines(discr, norm, col = "red", lwd = 2, lty = 2)
legend("topright", legend = c("Résidus", "Gaussienne"),
       col = c("blue", "red"), lwd = 2, lty = c(1, 2))

In [ ]:
# QQ-plot des résidus
qqnorm(erreurs, main = "QQ-plot des résidus")
qqline(erreurs, col = "red")

In [ ]:
# Test de Shapiro-Wilk
shapiro.test(erreurs)

**Conclusion :** p-value > 0.05 → on ne rejette pas la normalité des résidus.

### 1.3 Somme des carrés

In [ ]:
SCE = sum(erreurs^2)
SCT = sum((y - mean(y))^2)
SCM = sum((yajustes - mean(y))^2)
cat("SCE =", round(SCE, 2), "\nSCT =", round(SCT, 2), "\nSCM =", round(SCM, 2))
cat("\n\nVérification : SCM + SCE =", round(SCM + SCE, 2))

In [ ]:
# Coefficient de détermination
R2 = SCM / SCT
cat("R² =", round(R2, 4))
cat("\ncor(x,y)² =", round(cor(x, y)^2, 4))

### 1.4 Distribution de $\hat{\beta}_1$

On simule 100 échantillons du même modèle pour voir la variabilité de $\hat{\beta}_1$.

In [ ]:
set.seed(42)
age_sim = sample(40:56, replace = TRUE, 30000)
simx = matrix(age_sim, nrow = 100)
err_sim = rnorm(30000, 0, sqrt(SCE / (length(x) - 2)))
simerr = matrix(err_sim, nrow = 100)
simy = beta1 * simx + beta0 + simerr

simbeta = numeric(100)
for (i in 1:100) {
  simbeta[i] = cov(simx[i, ], simy[i, ]) / var(simx[i, ])
}

In [ ]:
# Comparaison avec Student
plot(density((simbeta - mean(simbeta)) / sqrt(var(simbeta))),
     main = "Distribution de β̂₁ standardisé", col = "blue", lwd = 2)
discr = seq(-3, 3, 0.1)
lines(discr, dt(discr, 298), col = "red", lwd = 2, lty = 2)
legend("topright", legend = c("Simulations", "Student(298)"),
       col = c("blue", "red"), lwd = 2, lty = c(1, 2))

### 1.5 Intervalle de confiance de $\beta_1$

In [ ]:
sbeta1 = sqrt(SCE / (398 * sum((x - mean(x))^2)))
binf = beta1 + qt(0.025, 298) * sbeta1
bsup = beta1 + qt(0.975, 298) * sbeta1
cat("IC 95% pour β₁ : [", round(binf, 4), ";", round(bsup, 4), "]")

hors_ic = simbeta[simbeta > bsup | simbeta < binf]
cat("\nValeurs hors IC :", length(hors_ic))

---
## 2. Tests de la Régression (Fisher / ANOVA)

In [ ]:
# Tableau d'ANOVA
anova(lm(y ~ x))

In [ ]:
# Statistique F et p-value
F_stat = 298 * SCM / SCE
cat("F observé =", round(F_stat, 2))
cat("\nQuantile F(0.95, 1, 298) =", round(qf(0.95, 1, 298), 4))
cat("\np-value =", 1 - pf(F_stat, 1, 298))

In [ ]:
# Résumé complet
summary(lm(y ~ x))

---
## 3. Prévision

Prédire la tension pour un homme de **50 ans** avec IC à 95%.

In [ ]:
se2 = SCE / 298
Y0 = beta1 * 50 + beta0
cat("Prévision Ŷ₀ =", round(Y0, 2), "mmHg")

bsup_prev = Y0 + qt(0.975, 298) * sqrt(se2 * (1 + 1/300 + (50 - mean(x))^2 / sum((x - mean(x))^2)))
binf_prev = Y0 - qt(0.975, 298) * sqrt(se2 * (1 + 1/300 + (50 - mean(x))^2 / sum((x - mean(x))^2)))
cat("\nIC prévision : [", round(binf_prev, 2), ";", round(bsup_prev, 2), "]")

In [ ]:
# Bande de confiance complète
plot(x, y, main = "Régression avec bandes de prévision",
     xlab = "Âge", ylab = "Tension", pch = 19, col = "gray70")
abline(beta0, beta1, col = "blue", lwd = 2)
discr = 40:56
predicted = beta1 * discr + beta0
bsup_band = predicted + qt(0.975, 298) * sqrt(se2 * (1 + 1/300 + (discr - mean(x))^2 / sum((x - mean(x))^2)))
binf_band = predicted - qt(0.975, 298) * sqrt(se2 * (1 + 1/300 + (discr - mean(x))^2 / sum((x - mean(x))^2)))
for (i in 1:length(discr)) {
  lines(c(discr[i], discr[i]), c(binf_band[i], bsup_band[i]), lwd = 3, col = "orange")
}

---
## 4. Régression Linéaire Multiple

On prédit `globale` (auto-évaluation auditive) par les seuils à 4 fréquences.

In [ ]:
audition = read.csv2("audition2.csv")
head(audition)
A5 = audition$A5
A10 = audition$A10
A20 = audition$A20
A40 = audition$A40
globale = audition$globale

In [ ]:
# Régression multiple
lm(globale ~ A5 + A10 + A20 + A40)

In [ ]:
# Tests statistiques complets
summary(lm(globale ~ A5 + A10 + A20 + A40))

**Interprétation :**
- **Fisher** : les variables ont globalement un effet significatif
- **Tous les tests de Student** significatifs → chaque variable contribue
- $R^2 \approx 0.97$ → 97% de la variance expliquée
- Le patient a une évaluation **lucide** de son état d'audition

---
## Résumé

| Concept | Fonction R |
|---------|------------|
| Régression simple | `lm(y ~ x)` |
| Résidus | `model$residuals` |
| ANOVA | `anova(lm())` |
| Prévision | Formule manuelle |
| Régression multiple | `lm(y ~ x1 + x2 + ...)` |